In [ ]:
import pandas as pd
fp = "../data/sba_loans_prepared/sba_train_raw_risky_nbrh.csv"
df = pd.read_csv(fp)

In [ ]:
df.LoanStatus.value_counts()

In [ ]:
loan_status = df.LoanStatus
cols = df.columns.tolist()
cols_to_drop = ["LoanStatus", "LoanID", "BorrName", "BankFDICNumber"]
cols_keep = [ c for c in cols if c not in cols_to_drop]

In [ ]:
df = df[cols_keep].reset_index(drop=True)

In [ ]:
num_cols = ["NumPmtsMade", "TermInMonths", "InitialInterestRate", "GrossChargeOffAmount", "GrossApproval"]
cat_cols = [ c for c in cols_keep if c not in num_cols ]

In [ ]:
dtypes = { c:"category" for c in cat_cols}
for c in num_cols:
    dtypes[c] = "float"

In [ ]:
#df = df.astype(dtypes)

In [ ]:
df.dtypes

In [ ]:
col_names = ["bor-" + str(i) for i in range(df.shape[0])]

In [ ]:
import gower_multiprocessing as gower

In [ ]:
distance_matrix = gower.gower_matrix(df)

In [ ]:
from sklearn.neighbors import kneighbors_graph
NUM_NBRS = 30
graph_adj_mat = kneighbors_graph(distance_matrix,n_neighbors=NUM_NBRS, mode="distance").toarray()

In [ ]:
graph_adj_mat

In [ ]:
import numpy as np
# Calculate the Degree Matrix (D)
A = graph_adj_mat
D_diag = np.sum(A, axis=1)
D = np.diag(D_diag)

# Calculate D^(-1/2)
D_inv_sqrt = np.diag(1.0 / np.sqrt(D_diag))

# Calculate the Normalized Laplacian (L)
I = np.identity(A.shape[0])
M = (D_inv_sqrt @ A )
L = I - (M @ D_inv_sqrt)

print("Normalized Laplacian Matrix:\n", L)

In [ ]:
# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(L)
eigenvalues = eigenvalues.real
eigenvectors = eigenvectors.real
# Get the indices that would sort the eigenvalues 
idx = eigenvalues.argsort()

# Sort the eigenvalues using the obtained indices
eigenvalues_sorted = eigenvalues[idx]

# Sort the eigenvectors using the same indices, applying them to the columns
eigenvectors_sorted = eigenvectors[:, idx]



In [ ]:
# 2. Set a threshold
threshold = 0.01

# 3. Create a boolean mask
mask = eigenvalues_sorted < threshold
true_count = np.count_nonzero(mask)
eigenvalues_sorted[:true_count]=0

In [ ]:
eigenvectors_sorted[:true_count]

In [ ]:
lam = 0
L_minus_lambda_I = L - lam * np.eye(L.shape[0])

# Calculate the rank
rank = np.linalg.matrix_rank(L_minus_lambda_I)

# Geometric multiplicity is n - rank
geometric_multiplicity = L.shape[0] - rank

print(f"Eigenvalue: {lam}")
print(f"Geometric Multiplicity: {geometric_multiplicity}")

In [ ]:
EVAL_LIMIT = 20
eig_val_idx = [(i+1) for i in range(2*EVAL_LIMIT)]
spec_gap = {"eig_val_index": eig_val_idx, "eig_vals": eigenvalues_sorted[:2*EVAL_LIMIT]}
df_sg = pd.DataFrame.from_dict(spec_gap, orient="columns")


In [ ]:
eigenvalues_sorted[:2*true_count]

In [ ]:
df_sg.head()

In [ ]:
import plotly.express as px
fig = px.scatter(df_sg, x="eig_val_index", y="eig_vals")
fig.show()

In [ ]:
from sklearn.manifold import SpectralEmbedding
NCOMP = 28
embedding = SpectralEmbedding(n_components=NCOMP, affinity="nearest_neighbors", n_neighbors=NUM_NBRS)
X_graph_emb = embedding.fit_transform(distance_matrix)

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=NCOMP, random_state=0, n_init="auto").fit(X_graph_emb)

In [ ]:
kmeans.labels_

In [ ]:
cols = ["G-" + str(i+1) for i in range(NCOMP)]
df_emb = pd.DataFrame(X_graph_emb, columns=cols)
df_emb["cluster"] = kmeans.labels_
df_emb["cluster"] = df_emb["cluster"].astype(str)

In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(df_emb[cols])


In [ ]:

df_tsne = pd.DataFrame(X_tsne, columns=['TSNE1', 'TSNE2'])
df_tsne['Cluster'] = kmeans.labels_ # Add cluster labels to the DataFrame
df_tsne['Cluster'] = df_tsne['Cluster'].astype(str)
fig = px.scatter(df_tsne, x='TSNE1', y='TSNE2', color='Cluster',
                 title='t-SNE Visualization with K-Means Clusters of SBA 7a Loan Data')
fig.show()